# Lab 00: How to set up the lab environment

**CPU | API key: `OPENAI_API_KEY` | one small model call**

## Goal

Check in one run that the core environment is ready: the packages at their pinned versions, the settings
and your key found, one model call answered, the shop's data in place, and the optional tools reported. Set the
repository up first with this lab's README (an Ubuntu VM with VS Code, GitHub Codespaces, or your own machine with
`uv` or `pip`), then run every cell from the top. When the last cell shows ✅ on every required line, you are ready
for Lab 01.

This is a checklist, not a lesson. The one cell worth understanding in detail is the first: every lab opens with it,
so Part A reads it line by line.

## What you will learn

- What the config cell at the top of every lab does: where it finds the model settings (`models.env`) and your key
  (`.env`), and what it prints.
- How to change the model for every lab with one line, or for your machine only.
- How to check that your packages match the repository's pins, that your key reaches the model, and that the shop's
  data is in place.
- Which tools and environments are optional, which lab needs each, and how to see whether you have them.
- How to read the summary at the end and fix a failed line from the troubleshooting table.

---

## Part A: Read the settings every lab starts from

Every lab opens with the same config cell. Before any other code runs, it answers three questions: where is the
repository folder, which model do we call, and where is the key? Run it; the explanation follows its output.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The cell printed the model, its reasoning effort and `key found: yes`. It never prints the key itself, only that it
found one. Here is what it did, in order:

1. **Find the repository folder.** `Path.cwd()` is the folder the notebook runs in, and `Path.cwd().parents` the
   folders above it, nearest first. `[Path.cwd(), *Path.cwd().parents]` puts them in one list, and `next(...)` returns
   the first folder in that list that holds `models.env`. That is why the same cell works whether the notebook runs in
   a lab folder or in the repository folder. If no folder holds `models.env` (the notebook was copied out of the
   repository), `next` has nothing to return and stops with `StopIteration`.
2. **Read the settings.** `dotenv_values` reads a file of `NAME=value` lines into a dictionary and changes nothing
   else. `settings` holds `models.env`; `local` holds your `.env`, or is empty if there is no `.env`.
3. **Let `.env` replace a setting.** The `settings.update(...)` line copies a value from `.env` only when its name is
   already in `models.env` and the value is not empty. So a `MODEL_DEFAULT=...` line in your `.env` changes the model
   on your machine only, and the key never enters `settings`.
4. **Find the key.** `os.environ.setdefault` keeps an `OPENAI_API_KEY` that is already in the environment (a
   Codespaces secret, for example) and otherwise sets the one from `.env`. The OpenAI library reads the key from the
   environment, so no later cell passes it or prints it.
5. **Stop early without a key.** `assert` stops the notebook here with its message, instead of three cells later
   with an authentication error.
6. **Name the two settings every call uses.** `MODEL` is `MODEL_DEFAULT`. `EFFORT` is `REASONING_EFFORT`, and
   `or None` turns an empty value into `None`, which Part C reads as "this model has no reasoning setting".

Later labs run this cell without the explanation. Here is the dictionary it built:

In [2]:
# Everything the labs read from models.env (and from .env, where it replaces a setting)
for name, value in settings.items():
    print(f"{name:32} {value}")

MODEL_DEFAULT                    gpt-6-luna
MODEL_STRONG                     gpt-6-luna
REASONING_EFFORT                 none
GEMINI_MODEL                     gemini-3.5-flash-lite
EMBEDDING_MODEL                  text-embedding-3-small
PRICE_INPUT_PER_MILLION          0.10
PRICE_CACHED_INPUT_PER_MILLION   0.01
PRICE_CACHE_WRITE_PER_MILLION    0.125
PRICE_OUTPUT_PER_MILLION         0.50
PRICE_EMBEDDING_PER_MILLION      0.02
PRICE_CONTAINER_SESSION_USD      0.03


These are the only settings the labs read. `MODEL_STRONG` is for the few tasks that need a more capable model,
`EMBEDDING_MODEL` turns text into vectors in the retrieval labs, and the five prices (US dollars per million tokens)
let a lab say what its calls cost. `GEMINI_MODEL` is read only by cells marked optional, and
`PRICE_CONTAINER_SESSION_USD` is the price of one hosted code-interpreter session in lab 17. None of them is a secret, which is why `models.env` is committed to the
repository and `.env` is not.

**To change the model for every lab,** change one line in `models.env`, `MODEL_DEFAULT=...`, and the prices under
it. **To change it on your machine only,** put the same line in your `.env`: the `update` line makes it win. A model
without a reasoning setting needs an empty `REASONING_EFFORT=` (Part C shows why). No notebook names a model in its
code, so nothing else changes.

---

## Part B: Check Python and the pinned packages

Every saved output in these labs came from the versions pinned in `pyproject.toml`. Another version can print a
different result, a warning or an error, so this part compares what the notebook's own Python has installed with
those pins. It reads the pins from `pyproject.toml` itself, so the check stays right when the pins move.

First, Python against the range the repository supports. `tomllib` (in the standard library since Python 3.11) reads
`pyproject.toml` into a dictionary; its `project` table holds `requires-python` and the list of `dependencies`.
`SpecifierSet` comes from `packaging`, the library `pip` uses to read version rules like `>=3.11,<3.15`.

In [3]:
import platform
import tomllib
from packaging.specifiers import SpecifierSet

project = tomllib.loads((ROOT / "pyproject.toml").read_text())["project"]
python_ok = platform.python_version() in SpecifierSet(project["requires-python"])
print("Python", platform.python_version(), "| the repository needs", project["requires-python"],
      "|", "inside the range" if python_ok else "outside the range")

Python 3.13.5 | the repository needs >=3.11,<3.15 | inside the range


A pass is a version inside the range. If it shows a Python you did not set up for this repository, the notebook is
running on another kernel (see the troubleshooting table at the end).

Then every pin. Each dependency is a `name==version` string, some with extras in brackets (`openai-agents[viz]` adds
the drawing support lab 07 uses), so the loop splits at `==` and drops the brackets. `version(name)` asks the
environment this notebook runs in, which is the one that matters: a terminal can use a different Python.

In [4]:
from importlib.metadata import version, PackageNotFoundError

different = []
for requirement in project["dependencies"]:
    name, pin = requirement.split("==")
    name = name.split("[")[0]                 # "openai-agents[viz]" is installed as "openai-agents"
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = "missing"
    if installed != pin:
        different.append(name)
    print(f"{name:28} pin {pin:10} installed {installed}")

print(len(project["dependencies"]), "pinned packages,", len(different), "different")

chromadb                     pin 1.5.9      installed 1.5.9
gradio                       pin 6.31.0     installed 6.31.0
ipykernel                    pin 7.4.0      installed 7.4.0
jupyterlab                   pin 4.6.5      installed 4.6.5
langchain                    pin 1.4.7      installed 1.4.7
langchain-core               pin 1.7.3      installed 1.7.3
langchain-openai             pin 1.7.3      installed 1.7.3
langgraph                    pin 1.2.19     installed 1.2.19
langgraph-checkpoint-sqlite  pin 3.1.3      installed 3.1.3
langsmith                    pin 0.14.16    installed 0.14.16
litellm                      pin 1.108.4    installed 1.108.4
matplotlib                   pin 3.11.2     installed 3.11.2
mcp                          pin 2.3.0      installed 2.3.0
mlflow                       pin 3.17.2     installed 3.17.2
openai                       pin 3.42.0     installed 3.42.0
openai-agents                pin 0.24.0     installed 0.24.0
pinecone                     p

A pass is `0 different`. A `missing` package, or another version, means the kernel is not the repository's `.venv`,
or the install did not finish. Do not upgrade a pinned package to silence a warning: the labs were written and run on
these versions. The list holds the packages the repository names directly; `uv` or `pip` installed what each of them
needs as well.

---

## Part C: Check that the key reaches the model

Parts A and B ran on this machine only. This part sends one request with your key and so checks three things at
once: the key is valid, `MODEL_DEFAULT` names a model your account can use, and this machine reaches the API. Lab 01
explains the call and everything in its response; here we read three fields.

⚠️ This cell sends one request billed to your key. It is under 50 tokens, a tiny fraction of a cent.

One argument needs a word. `reasoning_effort=EFFORT or omit` sends the effort from `models.env` when it is set. When
it is empty, `EFFORT` is `None`, and `omit` tells the OpenAI library to leave the field out of the request: a model
without reasoning refuses any request that names the field, even with no value in it.

In [5]:
from openai import OpenAI, omit

client = OpenAI()
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with one word: ready"}],
    max_completion_tokens=20,
    reasoning_effort=EFFORT or omit,
)

tokens_in, tokens_out = reply.usage.prompt_tokens, reply.usage.completion_tokens
cost = (tokens_in * float(settings["PRICE_INPUT_PER_MILLION"])
        + tokens_out * float(settings["PRICE_OUTPUT_PER_MILLION"])) / 1_000_000
print("reply:      ", reply.choices[0].message.content)
print("answered by:", reply.model)
print("tokens:     ", tokens_in, "in,", tokens_out, "out")
print(f"cost:        ${cost:.7f}")

reply:       ready
answered by: gpt-6-luna
tokens:      12 in, 4 out
cost:        $0.0000032


A pass is any short reply; ours was `ready`. `reply.model` names the model that answered, which can be a dated
version of the name in `models.env`. `usage` counts the tokens billed: 12 in for our five-word message (the
API adds a few formatting tokens around every message) and 4 out. The cost multiplies each
count by its price in `models.env`. If the call fails instead, the error names the problem: a wrong key, a model name
your account cannot use, or a reasoning setting the model does not accept (troubleshooting table).

---

## Part D: Check the shop's database and documents

Every lab works on one running case: the support desk of Chinook, a music shop. Its database is `data/chinook.db`
and its policies and support tickets are 36 short files in `data/corpus/`. Many checkpoints depend on their exact
contents, so this part counts what is there.

The labs only read the shop's data. `mode=ro` in the connection address opens the database read-only, so a mistake
in a later lab cannot change it; `uri=True` tells `sqlite3` to read the `file:...?mode=ro` address, and
`check_same_thread=False` lets the agent frameworks of later labs use the connection from another thread.

In [6]:
import sqlite3

db_path = ROOT / "data" / "chinook.db"
db = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, check_same_thread=False)
customers = db.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
invoices = db.execute("SELECT COUNT(*) FROM invoices").fetchone()[0]
db.close()
print("chinook.db:", customers, "customers,", invoices, "invoices")

chinook.db: 59 customers, 412 invoices


A pass is 59 customers and 412 invoices. Other counts mean the file was replaced or changed; get it back from the
repository (troubleshooting table).

The documents are the shop's 20 policies (`policy_*.md`) and 16 support tickets (`ticket_*.md`), which the retrieval
labs search.

In [7]:
corpus = sorted((ROOT / "data" / "corpus").glob("*.md"))
policies = [f for f in corpus if f.name.startswith("policy_")]
tickets = [f for f in corpus if f.name.startswith("ticket_")]
print("data/corpus:", len(corpus), "files,", len(policies), "policies,", len(tickets), "tickets")
print("first and last:", corpus[0].name, "...", corpus[-1].name)

data/corpus: 36 files, 20 policies, 16 tickets
first and last: policy_accounts.md ... ticket_16_overpayment_note.md


A pass is 36 files: 20 policies and 16 tickets.

---

## Part E: Check the optional tools, environments and keys

Most labs need nothing from this part. Each check reports what you have and which lab uses it, so a missing piece is
a choice, not a failure. The README says how to install each one.

**Graphviz.** Lab 07 draws an agent's structure with `dot`, a program from Graphviz. `shutil.which` looks for a
program on the system `PATH`, the way a terminal does, and `dot -V` prints its version (on `stderr`, which is why the
cell reads `stderr`).

In [8]:
import shutil
import subprocess

dot_found = shutil.which("dot") is not None
if dot_found:
    print(subprocess.run(["dot", "-V"], capture_output=True, text=True).stderr.strip())
else:
    print("dot: not found (optional; lab 07 draws agents with it)")

dot - graphviz version 15.1.1 (20260805.0921)


A pass is a version line like ours, `dot - graphviz version 15.1.1 (20260805.0921)`. Without Graphviz the line says `not found`, and lab 07's drawing cell
shows the `ExecutableNotFound` error from the troubleshooting table; only that drawing needs it.

**Docker.** Lab 17 can run model-written code inside a container, and it also runs without Docker. `docker info`
asks the Docker service (the daemon) for its version, and the service answers only when Docker is installed and
running.

In [9]:
if shutil.which("docker") is None:
    docker_ok = False
    print("docker: not installed (optional; lab 17's container runner)")
else:
    result = subprocess.run(["docker", "info", "--format", "{{.ServerVersion}}"], capture_output=True, text=True)
    docker_ok = result.returncode == 0
    print("docker server:", result.stdout.strip() if docker_ok else result.stderr.strip())

docker server: 29.9.2


A pass is a server version, ours `29.9.2`. If Docker is installed but not running, the line shows the
`failed to connect to the docker API` error from the troubleshooting table.

**Labs with their own environment.** Four labs use libraries that need other versions of packages the main
environment pins (an older `openai`, for example), so each keeps its own `pyproject.toml` and its own `.venv` in its
folder. You create one only when you do that lab (README, "Labs with their own environment"). The cell looks for each
folder's `.venv`.

In [10]:
OWN_ENVIRONMENTS = {
    "lab 08": "08_Google_ADK",
    "lab 14": "14_LlamaIndex_RAG",
    "bonus B4": "bonus/B4_CrewAI_Crew",
    "bonus B5": "bonus/B5_Microsoft_Agent_Framework",
}
own_ready = {}
for lab, folder in OWN_ENVIRONMENTS.items():
    own_ready[lab] = (ROOT / folder / ".venv").is_dir()
    print(f"{lab:9} {folder:36} {'set up' if own_ready[lab] else 'optional, not set up'}")

lab 08    08_Google_ADK                        optional, not set up
lab 14    14_LlamaIndex_RAG                    optional, not set up
bonus B4  bonus/B4_CrewAI_Crew                 optional, not set up
bonus B5  bonus/B5_Microsoft_Agent_Framework   optional, not set up


In our run none of the four separate environments was set up. These lines never fail: a lab's own environment matters only when you open that lab.

**Bonus keys.** Two bonus labs use services that need an account: Pinecone (B6) and LangSmith (B7). Their keys go
in `.env` beside the OpenAI key (`.env.example` has both lines). The cell says whether each is set, never its value.

In [11]:
BONUS_KEYS = {"PINECONE_API_KEY": "bonus B6", "LANGSMITH_API_KEY": "bonus B7"}
key_set = {}
for name, lab in BONUS_KEYS.items():
    key_set[name] = bool(os.environ.get(name) or local.get(name))
    print(f"{name:18} ({lab}): {'set' if key_set[name] else 'optional, not set'}")

PINECONE_API_KEY   (bonus B6): set
LANGSMITH_API_KEY  (bonus B7): set


Every core lab runs without these two keys.

---

## Check that you are ready

One line per check: the required ones pass or fail, and the optional ones say what is set up. If you skipped a cell,
this cell stops with a `NameError`; run that cell, then this one.

In [12]:
required = {
    "Python inside the range, every package at its pin": python_ok and not different,
    "models.env read, the key found": bool(MODEL and os.environ["OPENAI_API_KEY"]),
    "the model answers": bool(reply.choices[0].message.content),
    "chinook.db: 59 customers, 412 invoices": (customers, invoices) == (59, 412),
    "data/corpus: 20 policies, 16 tickets": (len(policies), len(tickets)) == (20, 16),
}
optional = {
    "Graphviz dot (lab 07)": dot_found,
    "Docker running (lab 17)": docker_ok,
    **{f"own environment, {lab}": ready for lab, ready in own_ready.items()},
    **{f"{name} ({lab})": key_set[name] for name, lab in BONUS_KEYS.items()},
}

print("Python", platform.python_version(), "|", reply.model)
for name, passed in required.items():
    print("✅" if passed else "❌", name)
print("\nOptional:")
for name, ready in optional.items():
    print("  ✅" if ready else "  --", name if ready else name + ": optional, not set up")

Python 3.13.5 | gpt-6-luna
✅ Python inside the range, every package at its pin
✅ models.env read, the key found
✅ the model answers
✅ chinook.db: 59 customers, 412 invoices
✅ data/corpus: 20 policies, 16 tickets

Optional:
  ✅ Graphviz dot (lab 07)
  ✅ Docker running (lab 17)
  -- own environment, lab 08: optional, not set up
  -- own environment, lab 14: optional, not set up
  -- own environment, bonus B4: optional, not set up
  -- own environment, bonus B5: optional, not set up
  ✅ PINECONE_API_KEY (bonus B6)
  ✅ LANGSMITH_API_KEY (bonus B7)


In our run every required line passed. Graphviz, Docker and both bonus keys were available; none of the four separate environments was set up. With every required line passing you are ready
for Lab 01. A failed line names the part to look at; the troubleshooting table below lists the causes we have
triggered and how to fix each.

### Checkpoint

| Check | A pass | Our run |
|---|---|---|
| Part A, config | `key found: yes`, and the model from `models.env` | `key found: yes`, reasoning effort `none` |
| Part B, packages | a Python inside `>=3.11,<3.15`; `0 different` | Python 3.13.5; 20 pinned packages, 0 different |
| Part C, model call | any short reply, and the model that answered | `ready`, 12 tokens in and 4 out |
| Part D, data | 59 customers, 412 invoices; 36 files (20 policies, 16 tickets) | 59 customers, 412 invoices; 36 files (20 policies, 16 tickets) |
| Part E, optional | a version line, or a line saying what is not set up | Graphviz 15.1.1, Docker 29.9.2 and both bonus keys; labs 08, 14, B4 and B5 environments not set up |

## What you can do now

- Say what the config cell at the top of every lab finds and prints, and why it can stop with `StopIteration` or an
  `AssertionError`.
- Change the model for every lab in `models.env`, or for your machine only in `.env`.
- Check in one run that your packages match the pins, that your key reaches the model and that the shop's data is in
  place.
- Tell which optional tools and environments you have, and which lab needs each.
- Find a failed line in the troubleshooting table and fix it.

## Troubleshooting

Each message below was triggered on purpose and copied from the run. The setup problems that come before the
notebook (installing `uv`, a blocked `pip`) are in the README.

| Where, and the message | Cause | Fix |
|---|---|---|
| Config cell: `ModuleNotFoundError: No module named 'dotenv'` | the notebook runs on a Python without the repository's packages | choose the repository's `.venv` as the kernel (README, "Choose the kernel"), then run from the top |
| Config cell: `StopIteration` | the notebook is not inside the repository folder (it was copied, downloaded alone or opened from elsewhere) | open it from the cloned or unzipped repository, where `models.env` sits |
| Config cell: `AssertionError: Put OPENAI_API_KEY in .env at the repository folder (copy .env.example).` | no `.env` beside `models.env`, or its `OPENAI_API_KEY=` line is empty | copy `.env.example` to `.env`, paste the key after `OPENAI_API_KEY=`, then run the config cell again |
| Part B: most packages `missing` and `openai` at `2.54.0` (`18 different` in our try) | the kernel is another lab's own environment (here lab 14's `.venv`), not the repository's | choose the `.venv` in the repository folder as the kernel |
| Part C: ``OpenAIError: Missing credentials. Please pass an `api_key`, ...`` | the config cell stopped without a key, and the cells after it were run anyway | fix the key (row above), then run from the config cell |
| Part C: `AuthenticationError: Error code: 401 - ... Incorrect API key provided ...` | the key is wrong, cut short or revoked, or an old `OPENAI_API_KEY` in the environment wins over `.env` | paste the key again; if it is set in the environment (a Codespaces secret, a shell profile), fix it there |
| Part C: `NotFoundError: Error code: 404 - The model ... does not exist or you do not have access to it.` | `MODEL_DEFAULT` is misspelt, or your account cannot use that model | check the line in `models.env`, and any `MODEL_DEFAULT` line in `.env` |
| Part C: `BadRequestError: Error code: 400 - Unrecognized request argument supplied: reasoning_effort` | `REASONING_EFFORT` is set for a model without reasoning | leave `REASONING_EFFORT=` empty for that model |
| Part D: `OperationalError: unable to open database file` | `data/chinook.db` is missing or was moved | restore it from the repository: `git checkout -- data/chinook.db`, or unzip the ZIP again |
| Part E: `dot: not found`; in lab 07, `ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH` | Graphviz is not installed, or not on the `PATH` | install Graphviz (README, "Optional tools"), then restart the kernel |
| Part E: `failed to connect to the docker API at unix:///...; check if the path is correct and if the daemon is running` | Docker is installed but not running, or `DOCKER_HOST` points to a socket that does not exist | start Docker Desktop, or on Linux `sudo systemctl start docker`; unset a stale `DOCKER_HOST`; then run the cell again |

## Resources

- [uv: working on projects](https://docs.astral.sh/uv/guides/projects/): `uv sync`, `uv run` and the lock file
- [python-dotenv](https://pypi.org/project/python-dotenv/): `dotenv_values` and the `.env` file format
- [Python `tomllib`](https://docs.python.org/3/library/tomllib.html) and
  [`importlib.metadata`](https://docs.python.org/3/library/importlib.metadata.html): reading `pyproject.toml` and the
  installed versions in Part B
- [Python `sqlite3`: SQLite URIs](https://docs.python.org/3/library/sqlite3.html#how-to-work-with-sqlite-uris): the
  read-only `mode=ro` address in Part D
- [OpenAI: models](https://developers.openai.com/api/docs/models) and
  [pricing](https://developers.openai.com/api/docs/pricing): the names and prices you can put in `models.env`

Next: [Lab 01: How to call a model and read the response](../01_Model_Call/README.md). Back to
[the lab list](../README.md).